# Week 1: Research Question & Problem Framing (ML-02)

**Track:** Applied Search Intelligence: Google Search Ranking & Discoverability  
**Company / Problem:** FlyRank Content Refresh Opportunity System  
**Author:** Altaf (ML Intern)  
**Dataset:** Starter Anonymized Release (`data/raw/content_refresh_anonymized.csv`)  
**Evaluation Metric:** Precision@50 (vs 54.2% Base Rate)  

---

## 1. Executive Summary & Problem Context

FlyRank builds and operates content as infrastructure: algorithmic systems research, write, publish, and continuously observe performance across thousands of published pages. However, once content ranks on Google, it quietly undergoes **organic search decay**—rankings slip, impressions erode, and click volume drops over time. 

Most editorial and SEO teams discover decay months too late, after traffic has evaporated. Conversely, reviewing every existing page manually is impossible: a typical client portfolio contains thousands of URLs, whereas human editorial capacity is strictly bounded (typically 20–50 pages per week).

**The Core Question:** *Out of thousands of aging articles, which ones should an editorial team refresh FIRST to protect high-intent search traffic?*

## 2. The Four Framing Questions

### 2.1 What decision does this improve?
This model does not aim to passively "predict Google's algorithm" or forecast exact future traffic volumes. Instead, it improves an operational triage decision: **allocating scarce human editorial and optimization bandwidth** to the specific URLs where intervention will prevent the greatest organic click loss.

### 2.2 Who acts on the output, and what do they do?
- **The Actor:** FlyRank content strategists, SEO managers, and client editorial reviewers.
- **The Action:** Reviewers inspect the top-ranked queue items and execute targeted updates based on attached reason codes:
  - `refresh_and_review_ctr`: Rewrite SERP title tags and meta descriptions to recapture lost CTR.
  - `refresh_and_review_engagement`: Improve above-the-fold layout, readability, and media assets to curb bounce rates.
  - `expand_and_refresh`: Expand thin articles (<1,200 words) that retain Google visibility.
  - `refresh`: Update outdated facts, figures, internal links, and timestamps.
  - `monitor`: Maintain steady state without wasting editorial budget.

### 2.3 What does a wrong answer cost?
- **Cost of a False Positive (Ranking a healthy/irrelevant page in Top 50):** Wasted human editorial labor (~1.5–2 hours per page) updating content that did not need intervention, while ignoring truly decaying assets.
- **Cost of a False Negative (Failing to flag a decaying high-demand asset):** Permanent loss of prime SERP positions (e.g., slipping from position #3 to #14), causing hundreds of monthly high-intent organic clicks to migrate to competitors.

### 2.4 Why does data/ML help over fixed rules?
FlyRank currently uses hand-written heuristic rules (e.g., `if days_since_update >= 180 and impressions >= 500`). While intuitive, fixed rules fail in search intelligence because:
1. **Signal entanglement:** Freshness interacts non-linearly with search position, CTR, intent, and scroll depth.
2. **Niche variance:** An article in medical/finance decays at a radically different velocity than one in B2B SaaS.
3. **Capacity mismatch:** Hand rules either flood editors with hundreds of unranked flags or miss subtle early-warning decay signatures.

## 3. Mapping to ML Task Type

| Dimensional Axis | Formal Specification |
|---|---|
| **Task Type** | Decision-Support Priority Ranking / Calibrated Binary Risk Scoring |
| **Unit of Analysis** | One published content asset (`content_id`), grouped by client domain (`client_id`) |
| **Feature Observation Window** | Trailing 90-day search console and analytics telemetry |
| **Target Definition** | `is_declining_label` $\in \{0, 1\}$ (Observed $\ge 15\%$ organic click decline in forward 30-day evaluation horizon) |
| **Base Rate** | **54.2%** (16,262 declining items out of 30,000 total rows in starter release) |
| **Selection Metric** | **Precision@50** (Precision within the first 50 prioritized queue items) |
| **Secondary Metrics** | ROC AUC, Average Precision (PR AUC), Recall, F1 |
| **Validation Strategy** | Grouped Client Holdout (Never splitting rows randomly across the same client) |

## 4. The One-Paragraph Canonical Frame

> **For FlyRank content strategists and client editorial teams**, deciding **which aging search articles to review and refresh first**, we will build a **decision-support priority ranker** from **trailing 90-day search console and behavioral analytics telemetry across 32 clients**, scoring the probability of **forward 30-day click decline (`is_declining_label`)** measured by **Precision@50** against a 54.2% base rate. A wrong call costs **wasted editorial review labor ($45–$70/hr) on stable pages or irreversible loss of Page 1 organic visibility**. A plain hand rule isn't enough because **decay dynamics involve complex, non-linear interactions between position, staleness, CTR, and scroll depth that vary across domains**. We will claim only **observed, measured, directional, and decision-support** results.

## 5. Dataset Summary & Ground Truth Checks

| Metric | Observed Value |
|---|---|
| Total Rows | 30,000 |
| Total Columns | 44 |
| Distinct Pseudonymized Clients | 32 |
| Declining Rows (`is_declining_label = 1`) | 16,262 |
| Declining Base Rate | 54.2% |
| Client ID Usage | Grouping / Split variable only (Never feature input) |
| Metric Units | Percentages are $\times 100$ (`ctr = 0.76` means $0.76\%$) |

## 6. Verification & Rubric Checklist (ML-02)

- [x] **Decision identified:** Triage priority queue for content refresh.
- [x] **Actor identified:** Content strategists and editorial reviewers.
- [x] **Action identified:** Targeted updates mapped to reason codes (`refresh_and_review_ctr`, etc.).
- [x] **Cost of error quantified:** Wasted manual audit hours vs lost organic SERP clicks.
- [x] **Why ML helps stated:** Non-linear signal interaction across freshness, CTR, and position.
- [x] **Unit of analysis defined:** `content_id` grouped by `client_id`.
- [x] **Target is observed:** Measured click decline in forward window, not an arbitrary hand rule.
- [x] **Metric selected before modeling:** Precision@50 reported alongside the 54.2% base rate.
- [x] **Honest claim language used:** Observed / measured / directional / decision-support.